In [ ]:
import os
import json
import requests
from typing import TypedDict, Annotated, List, Literal
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from pydantic import BaseModel, Field


load_dotenv('')

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
GITHUB_REPO  = "rahul8879/e-comm-agentic-demo"
HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}
BASE_URL = f"https://api.github.com/repos/{GITHUB_REPO}"

llm = ChatOpenAI(model="gpt-4o", temperature=0)

r = requests.get(BASE_URL, headers=HEADERS)
print(f"Repo: {r.json().get('full_name')}")
print(f"Status: {r.status_code}")


/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo: rahul8879/e-comm-agentic-demo
Status: 200


In [4]:
def get_pr_details(pr_number: int) -> dict:
    r = requests.get(f"{BASE_URL}/pulls/{pr_number}", headers=HEADERS)
    data = r.json()
    return {
        "title":  data.get("title"),
        "author": data.get("user", {}).get("login"),
        "body":   data.get("body") or "No description provided.",
        "branch": data.get("head", {}).get("ref"),
    }

def get_pr_files(pr_number: int) -> list:
    r = requests.get(f"{BASE_URL}/pulls/{pr_number}/files", headers=HEADERS)
    files = r.json()
    return [
        {"filename": f["filename"], "patch": f.get("patch", "")}
        for f in files
    ]

def post_pr_comment(pr_number: int, comment_body: str) -> str:
    url = f"{BASE_URL}/issues/{pr_number}/comments"
    r = requests.post(url, headers=HEADERS, json={"body": comment_body})
    return "posted" if r.status_code == 201 else f"failed: {r.status_code}"



# get_pr_files(1)

In [5]:
# shared state
class ReviewState(TypedDict):
    pr_number: int
    pr_details: dict
    pr_files: list
    plan: List[str]              # which specialists the Manager decided to call
    security_findings: str
    style_findings: str
    test_findings: str
    final_comment: str
    approved: bool

In [6]:
#Manager (Supervisor) node

class ReviewPlan(BaseModel):
    needs_security: bool = Field(description="True if this PR touches auth, payments, SQL, secrets, or user input handling")
    needs_style: bool = Field(description="True if this PR changes naming, formatting, or structure worth a style pass")
    needs_tests: bool = Field(description="True if this PR adds or changes logic that should have test coverage")
    reason: str = Field(description="One short sentence explaining the routing decision")


planner_llm = llm.with_structured_output(ReviewPlan)



In [7]:

def manager_node(state: ReviewState) -> dict:
    pr_details = get_pr_details(state["pr_number"])
    pr_files = get_pr_files(state["pr_number"])

    diff_summary = "\n\n".join(
        f"File: {f['filename']}\n{f['patch'][:800]}" for f in pr_files
    )

    plan: ReviewPlan = planner_llm.invoke([
        SystemMessage(content=(
            "You are the lead reviewer on CodeSentinel. Look at this PR's diff "
            "and decide which specialist reviewers actually need to look at it. "
            "Don't call a specialist unless their concern is genuinely relevant."
        )),
        HumanMessage(content=f"PR title: {pr_details['title']}\n\nDiff:\n{diff_summary}")
    ])

    chosen = []
    if plan.needs_security: chosen.append("security")
    if plan.needs_style:    chosen.append("style")
    if plan.needs_tests:    chosen.append("tests")

    print(f"Manager's plan: {chosen}  —  {plan.reason}")

    return {
        "pr_details": pr_details,
        "pr_files": pr_files,
        "plan": chosen,
    }


In [8]:
# ReviewState(pr_number=1, pr_details={}, pr_files=[], plan=[], security_findings="", style_findings="", test_findings="", final_comment="", approved=False)

In [9]:
# manager_node({"pr_number": 1})

In [10]:
class ReviewState(TypedDict):
    pr_number: int
    pr_details: dict
    pr_files: list
    plan: List[str]              # which specialists the Manager decided to call
    security_findings: str
    style_findings: str
    test_findings: str
    final_comment: str
    approved: bool

In [11]:
def _diff_text(state: ReviewState) -> str:
    return "\n\n".join(
        f"File: {f['filename']}\n{f['patch'][:1000]}" for f in state["pr_files"]
    )

def security_node(state: ReviewState) -> dict:
    response = llm.invoke([
        SystemMessage(content=(
            "You are a security specialist. ONLY look for: SQL injection, "
            "hardcoded secrets, missing auth checks, unsafe input handling. "
            "Ignore style and naming completely. Be specific about file and line."
        )),
        HumanMessage(content=_diff_text(state))
    ])
    return {"security_findings": response.content}

In [12]:
def style_node(state: ReviewState) -> dict:
    response = llm.invoke([
        SystemMessage(content=(
            "You are a code style specialist. ONLY look for: naming conventions, "
            "formatting, function length, and readability. Ignore security and "
            "test coverage completely."
        )),
        HumanMessage(content=_diff_text(state))
    ])
    return {"style_findings": response.content}

In [13]:
def tests_node(state: ReviewState) -> dict:
    response = llm.invoke([
        SystemMessage(content=(
            "You are a test-coverage specialist. ONLY look for: new logic that "
            "has no matching test, and edge cases that look untested. Ignore "
            "security and style completely."
        )),
        HumanMessage(content=_diff_text(state))
    ])
    return {"test_findings": response.content}

In [14]:
def route_to_specialists(state: ReviewState):
    targets = []
    if "security" in state["plan"]: targets.append("security")
    if "style" in state["plan"]:    targets.append("style")
    if "tests" in state["plan"]:    targets.append("tests")
    return targets if targets else ["aggregator"]


In [15]:
class CodeReviewResult(BaseModel):
    verdict: Literal["APPROVE", "NEEDS_CHANGES"] = Field(description="Final call")
    severity: Literal["LOW", "MEDIUM", "HIGH", "CRITICAL"] = Field(description="Worst issue found")
    summary: str = Field(description="2-3 sentence combined summary for the PR author")

aggregator_llm = llm.with_structured_output(CodeReviewResult)




def aggregator_node(state: ReviewState) -> dict:
    findings = []
    if state.get("security_findings"): findings.append(f"SECURITY:\n{state['security_findings']}")
    if state.get("style_findings"):    findings.append(f"STYLE:\n{state['style_findings']}")
    if state.get("test_findings"):     findings.append(f"TESTS:\n{state['test_findings']}")

    combined = "\n\n".join(findings) if findings else "No specialist flagged anything. This PR was low-risk."

    result: CodeReviewResult = aggregator_llm.invoke([
        SystemMessage(content=(
            "You are the lead reviewer. Combine the specialist findings below into "
            "ONE final verdict for the PR author. Don't just concatenate — synthesize."
        )),
        HumanMessage(content=combined)
    ])

    comment = (
        f"## CodeSentinel Review — {result.verdict}\n"
        f"**Severity:** {result.severity}\n\n"
        f"{result.summary}\n\n"
        f"---\n{combined}"
    )
    return {"final_comment": comment}
    

In [16]:
#Human approval + post

def approval_node(state: ReviewState) -> dict:
    print(state["final_comment"])
    answer = input("\nPost this comment to the real PR? (y/n): ").strip().lower()
    return {"approved": answer == "y"}

def post_node(state: ReviewState) -> dict:
    if state["approved"]:
        status = post_pr_comment(state["pr_number"], state["final_comment"])
        print(f"Post status: {status}")
    else:
        print("Skipped — not posted.")
    return {}


In [17]:
graph = StateGraph(ReviewState)


graph.add_node("manager", manager_node)
graph.add_node("security", security_node)
graph.add_node("style", style_node)
graph.add_node("tests", tests_node)
graph.add_node("aggregator", aggregator_node)
graph.add_node("approval", approval_node)
graph.add_node("post", post_node)


graph.add_edge(START, "manager")
graph.add_conditional_edges("manager", route_to_specialists, ["security", "style", "tests", "aggregator"])

graph.add_edge("security", "aggregator")
graph.add_edge("style", "aggregator")
graph.add_edge("tests", "aggregator")
graph.add_edge("aggregator", "approval")

graph.add_edge("approval", "post")
graph.add_edge("post", END)

app = graph.compile()

In [35]:
# app

In [19]:
PR_NUMBER = 11

result = app.invoke({
    "pr_number": PR_NUMBER,
    "pr_details": {},
    "pr_files": [],
    "plan": [],
    "security_findings": "",
    "style_findings": "",
    "test_findings": "",
    "final_comment": "",
    "approved": False,
})


Manager's plan: ['security']  —  The PR contains a hardcoded password and potential SQL injection vulnerability.
## CodeSentinel Review — NEEDS_CHANGES
**Severity:** CRITICAL

The code contains critical security vulnerabilities, including SQL injection risks, lack of authentication checks, and hardcoded secrets. These issues must be addressed to ensure the application is secure. Use parameterized queries to prevent SQL injection, implement proper authentication and authorization checks, and remove any hardcoded sensitive information from the code.

---
SECURITY:
In the provided code snippet from `orders.py`, there are several security issues:

1. **SQL Injection (Line 48):**
   - The query construction `query = f"SELECT * FROM orders WHERE id = somethign bad"` is vulnerable to SQL injection. The `order_id` should be parameterized to prevent injection attacks. Instead of directly interpolating the `order_id` into the SQL query string, use parameterized queries or prepared statements pro